<a href="https://colab.research.google.com/github/EddyferO/Smart-Glass-Medical-Assistant/blob/main/Model_testing_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SETUP

In [1]:
!pip install -q nltk bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s eta 0:00:00


MODEL 1 — LONG-T5

In [3]:
import torch

from transformers import AutoTokenizer, LongT5ForConditionalGeneration

print("Loading Long-T5...")
tokenizer_lt5 = AutoTokenizer.from_pretrained("google/long-t5-tglobal-base")
model_lt5 = LongT5ForConditionalGeneration.from_pretrained(
    "google/long-t5-tglobal-base",
    torch_dtype=torch.float16
).to("cuda")
print("Long-T5 loaded!")

def summarize_long_t5(text, max_words=150):
    prompt = (
        "Generate a detailed abstractive summary of the following medical text. "
        "Do not copy sentences directly. Rewrite the key information in your own words, "
        "covering indications, dosing, warnings, side effects, and drug interactions: "
        + text
    )
    inputs = tokenizer_lt5(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=16384
    ).to(model_lt5.device)
    output_ids = model_lt5.generate(
        inputs["input_ids"],
        min_new_tokens=100,
        max_new_tokens=300,
        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=4,
        early_stopping=False
    )
    return tokenizer_lt5.decode(output_ids[0], skip_special_tokens=True)

Loading Long-T5...


pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/297 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Long-T5 loaded!


MODEL 2 — MEDGEMMA

In [4]:
import warnings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("Loading MedGemma...")
tokenizer_mg = AutoTokenizer.from_pretrained("google/medgemma-4b-it")
model_mg = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-4b-it",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
model_mg.generation_config.max_length = None
pipe_mg = pipeline("text-generation", model=model_mg, tokenizer=tokenizer_mg, max_new_tokens=400)
print("MedGemma loaded!")

def chunk_text(text, chunk_size=300):
    """Split text into chunks of roughly chunk_size words."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i + chunk_size]))
    return chunks

def medgemma_summarize_chunk(chunk):
    """Summarize a single chunk."""
    messages = [{
        "role": "user",
        "content": (
            "Summarize this medical text in 3-4 sentences in your own words, "
            "capturing the key clinical information:\n\n" + chunk
        )
    }]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

def medgemma_base_prompt(text, max_words=150):
    """Chunk the text, summarize each chunk, then combine into final summary."""
    chunks = chunk_text(text, chunk_size=300)

    # If text is short enough, summarize directly
    if len(chunks) == 1:
        return medgemma_summarize_chunk(chunks[0])

    # Summarize each chunk
    chunk_summaries = [medgemma_summarize_chunk(chunk) for chunk in chunks]
    combined = " ".join(chunk_summaries)

    # Final pass to combine into one clean paragraph
    messages = [{
        "role": "user",
        "content": (
            f"Combine these summaries into one coherent medical summary paragraph "
            f"of {max_words} words or less. Do not repeat information:\n\n" + combined
        )
    }]
    formatted = tokenizer_mg.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        output = pipe_mg(formatted, return_full_text=False)
    return output[0]["generated_text"].strip()

Loading MedGemma...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


MedGemma loaded!


In [5]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

EVALUATION

In [7]:
import time
import warnings
import textwrap
import logging
from bert_score import score as bert_score

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

test_cases = [
    {
        "label": "EPIDIOLEX — Indications, Dosing & Warnings",
        "text": """EPIDIOLEX is indicated for the treatment of seizures associated with Lennox-Gastaut
        syndrome, Dravet syndrome, or tuberous sclerosis complex in patients 1 year of age and older.
        The recommended starting dosage is 2.5 mg/kg by mouth twice daily. After one week, the dosage
        can be increased to a maintenance dosage of 5 mg/kg twice daily. Based on individual clinical
        response and tolerability, EPIDIOLEX can be increased up to a maximum recommended maintenance
        dosage of 10 mg/kg twice daily for LGS and DS, or 12.5 mg/kg twice daily for TSC.
        Dosage adjustment is recommended for patients with moderate or severe hepatic impairment.
        EPIDIOLEX can cause transaminase elevations. Concomitant use of valproate and higher doses
        of EPIDIOLEX increase the risk of transaminase elevations. EPIDIOLEX can cause somnolence
        and sedation. Antiepileptic drugs including EPIDIOLEX increase the risk of suicidal thoughts
        or behavior. EPIDIOLEX should be gradually withdrawn to minimize the risk of increased seizure
        frequency and status epilepticus.""",
        "reference": """EPIDIOLEX is a cannabidiol oral solution indicated for seizures associated with
        Lennox-Gastaut syndrome, Dravet syndrome, and tuberous sclerosis complex in patients aged 1
        and older. Starting dose is 2.5 mg/kg twice daily, with maintenance up to 10 mg/kg twice daily
        for LGS and DS, or 12.5 mg/kg twice daily for TSC. Dose adjustment is needed for moderate or
        severe hepatic impairment. Key warnings include transaminase elevations especially with
        valproate, somnolence, suicidal ideation, and risk of seizures upon abrupt withdrawal."""
    },
    {
        "label": "EPIDIOLEX — Adverse Reactions & Drug Interactions",
        "text": """The most common adverse reactions in patients with Lennox-Gastaut syndrome or Dravet
        syndrome are somnolence, decreased appetite, diarrhea, transaminase elevations, fatigue,
        malaise, asthenia, rash, insomnia, and infections. In controlled studies, the incidence of
        somnolence and sedation was 32% in EPIDIOLEX-treated patients compared with 11% on placebo.
        The incidence of ALT elevations above 3 times the upper limit of normal was 13% in
        EPIDIOLEX-treated patients compared with 1% in patients on placebo. Concomitant use of
        EPIDIOLEX with clobazam increases plasma concentrations of N-desmethylclobazam, the active
        metabolite of clobazam, which may increase the risk of clobazam-related adverse reactions.
        Concomitant use of EPIDIOLEX and valproate increases the incidence of liver enzyme elevations.
        Strong inducers of CYP3A4 or CYP2C19 decrease cannabidiol plasma concentrations by
        approximately 32%. Cannabidiol is metabolized in the liver primarily by CYP2C19 and CYP3A4
        enzymes. EPIDIOLEX interacts with multiple drug metabolizing enzymes including CYP1A2,
        CYP2B6, CYP2C8, CYP2C19, and UGT1A9.""",
        "reference": """Common adverse reactions for EPIDIOLEX include somnolence at 32% versus 11%
        for placebo, decreased appetite, diarrhea, transaminase elevations at 13% versus 1% for
        placebo, fatigue, rash, insomnia, and infections. Clobazam co-administration raises
        N-desmethylclobazam levels and increases adverse reaction risk. Valproate co-use raises
        liver enzyme elevations. Strong CYP3A4 or CYP2C19 inducers reduce cannabidiol levels by
        about 32%. Cannabidiol is primarily metabolized by CYP2C19 and CYP3A4 and interacts with
        CYP1A2, CYP2B6, CYP2C8, and UGT1A9."""
    },
    {
        "label": "EPIDIOLEX — Clinical Trial Efficacy Results",
        "text": """The effectiveness of EPIDIOLEX for Lennox-Gastaut syndrome was established in two
        randomized double-blind placebo-controlled trials in patients aged 2 to 55 years. In Study 1,
        the median percent reduction in drop seizure frequency was 44% for EPIDIOLEX 20 mg/kg/day
        compared to 22% for placebo. In Study 2, the median percent reduction was 37% for the
        10 mg/kg/day group and 42% for the 20 mg/kg/day group compared to 17% for placebo.
        The effectiveness for Dravet syndrome was demonstrated in a single randomized double-blind
        placebo-controlled trial in 120 patients aged 2 to 18 years. The median percent reduction
        in convulsive seizures was 39% for EPIDIOLEX 20 mg/kg/day compared to 13% for placebo.
        For tuberous sclerosis complex, the median percentage change from baseline in TSC-associated
        seizure frequency was 43% reduction for EPIDIOLEX 25 mg/kg/day compared to 20% reduction
        for placebo. A reduction in seizures was observed within 4 weeks of initiating treatment
        with EPIDIOLEX across all three conditions.""",
        "reference": """Two double-blind placebo-controlled RCTs established EPIDIOLEX efficacy for
        LGS, showing 44% and 42% median drop seizure reductions versus 22% and 17% for placebo.
        One RCT in 120 Dravet syndrome patients showed a 39% reduction in convulsive seizures
        versus 13% for placebo. For TSC, one RCT showed a 43% reduction in seizures versus 20%
        for placebo. Seizure reduction was observed within 4 weeks across all three indications."""
    }
]

def wrap_output(text, width=80, indent="  "):
    return "\n".join(textwrap.wrap(text.strip(), width=width, initial_indent=indent, subsequent_indent=indent))

def get_bertscore(reference, hypothesis):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _, _, F1 = bert_score([hypothesis], [reference], lang="en", verbose=False)
    return round(F1[0].item(), 4)

lt5_times, mg_times = [], []
lt5_bert,  mg_bert  = [], []

print("\n" + "="*70)
print("        SUMMARY COMPARISON: Long-T5  vs  MedGemma")
print("="*70)

for i, case in enumerate(test_cases, 1):
    print(f"\n{'─'*70}")
    print(f"  TEST CASE {i}: {case['label']}")
    print(f"{'─'*70}")

    print(f"\n  ORIGINAL TEXT:")
    print(wrap_output(case['text']))

    print(f"\n  LONG-T5 SUMMARY:")
    start = time.time()
    lt5_output = summarize_long_t5(case["text"], max_words=150)
    lt5_elapsed = round(time.time() - start, 2)
    lt5_times.append(lt5_elapsed)
    b1 = get_bertscore(case["reference"], lt5_output)
    lt5_bert.append(b1)
    print(wrap_output(lt5_output))
    print(f"\n  ⏱ Time: {lt5_elapsed}s  |  BERTScore F1: {b1}")

    print(f"\n  MEDGEMMA SUMMARY:")
    start = time.time()
    mg_output = medgemma_base_prompt(case["text"], max_words=150)
    mg_elapsed = round(time.time() - start, 2)
    mg_times.append(mg_elapsed)
    b2 = get_bertscore(case["reference"], mg_output)
    mg_bert.append(b2)
    print(wrap_output(mg_output))
    print(f"\n  ⏱ Time: {mg_elapsed}s  |  BERTScore F1: {b2}")

avg_lt5_bert = round(sum(lt5_bert)  / len(lt5_bert),  4)
avg_mg_bert  = round(sum(mg_bert)   / len(mg_bert),   4)
avg_lt5_time = round(sum(lt5_times) / len(lt5_times), 2)
avg_mg_time  = round(sum(mg_times)  / len(mg_times),  2)
winner       = "Long-T5"  if avg_lt5_bert > avg_mg_bert  else "MedGemma"
faster       = "Long-T5"  if avg_lt5_time < avg_mg_time  else "MedGemma"

print(f"\n{'='*70}")
print("  OVERALL COMPARISON")
print(f"{'='*70}")
print(f"  {'Metric':<25} {'Long-T5':>10} {'MedGemma':>10}")
print(f"  {'─'*45}")
print(f"  {'Avg BERTScore F1':<25} {avg_lt5_bert:>10} {avg_mg_bert:>10}")
print(f"  {'Avg Time (s)':<25} {avg_lt5_time:>10} {avg_mg_time:>10}")
print(f"{'─'*70}")
print(f"  More Accurate Model  : {winner}")
print(f"  Faster Model         : {faster}")
print(f"{'='*70}")


        SUMMARY COMPARISON: Long-T5  vs  MedGemma

──────────────────────────────────────────────────────────────────────
  TEST CASE 1: EPIDIOLEX — Indications, Dosing & Warnings
──────────────────────────────────────────────────────────────────────

  ORIGINAL TEXT:
  EPIDIOLEX is indicated for the treatment of seizures associated with Lennox-
  Gastaut          syndrome, Dravet syndrome, or tuberous sclerosis complex in
  patients 1 year of age and older.          The recommended starting dosage is
  2.5 mg/kg by mouth twice daily. After one week, the dosage          can be
  increased to a maintenance dosage of 5 mg/kg twice daily. Based on individual
  clinical          response and tolerability, EPIDIOLEX can be increased up to
  a maximum recommended maintenance          dosage of 10 mg/kg twice daily for
  LGS and DS, or 12.5 mg/kg twice daily for TSC.          Dosage adjustment is
  recommended for patients with moderate or severe hepatic impairment.
  EPIDIOLEX can cause tra

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  EPIDIOLEX, a medication for Lennox-Gastaut syndrome and Dravet syndrome, can
  cause common side effects like somnolence, diarrhea, and fatigue.  It
  significantly increases the risk of somnolence and liver enzyme elevations
  compared to a placebo.  It also interacts with other medications, such as
  clobazam and valproate, to increase the risk of side effects and affect
  cannabidiol's levels in the body.  These interactions are mediated by various
  drug-metabolizing enzymes in the liver.

  ⏱ Time: 9.92s  |  BERTScore F1: 0.8391

──────────────────────────────────────────────────────────────────────
  TEST CASE 3: EPIDIOLEX — Clinical Trial Efficacy Results
──────────────────────────────────────────────────────────────────────

  ORIGINAL TEXT:
  The effectiveness of EPIDIOLEX for Lennox-Gastaut syndrome was established in
  two          randomized double-blind placebo-controlled trials in patients
  aged 2 to 55 years. In Study 1,          the median percent reduction in drop
 

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

  Rewrite the key information in your own words, covering indications, dosing,
  warnings, side effects, and drug interactions: The effectiveness of EPIDIOLEX
  for Lennox-Gastaut syndrome was established in two randomized double-blind
  placebo-controlled trials in patients aged 2 to 55 years.In Study 1, the
  median percent reduction in drop seizure frequency was 44% for EPIDIOLEDEX 20
  mg/kg/day compared to 22% for placebo.In Study 2, the median reduction was 37%
  for the 10 mg/KG/day group and 42% for the 20 mg/Kg/day group compared to 17%
  for placebo.A reduction in seizures was observed within 4 weeks of initiating
  treatment with EPIDIOLETX across all three conditions.

  ⏱ Time: 5.69s  |  BERTScore F1: 0.8484

  MEDGEMMA SUMMARY:


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

  EPIDIOLEX is an effective treatment for Lennox-Gastaut syndrome, Dravet
  syndrome, and tuberous sclerosis complex (TSC).  Clinical trials showed a
  significant reduction in seizure frequency across all three conditions, with
  the greatest reduction observed in Lennox-Gastaut syndrome.  These
  improvements were demonstrated within 4 weeks of starting EPIDIOLEX.

  ⏱ Time: 6.5s  |  BERTScore F1: 0.8589

  OVERALL COMPARISON
  Metric                       Long-T5   MedGemma
  ─────────────────────────────────────────────
  Avg BERTScore F1              0.8426     0.8442
  Avg Time (s)                    7.19       9.08
──────────────────────────────────────────────────────────────────────
  More Accurate Model  : MedGemma
  Faster Model         : Long-T5
